# Component Usage Demo
Demo how to use the component library, and extract important components related to embedded platform energy, latency, area analysis.

In [1]:
import hwcomponents as hwc

# Find and list all installed component models
print(f'Listing component models')
all_installed_models = hwc.get_models()
print(f'Number of available component models: {len(all_installed_models)}')
# A list of all available component models
print(f'All available component models: {[model.__name__ for model in all_installed_models]}')
# Show example of the first 5 models and their supported component types
# for model in all_installed_models[:5]:
#     print(f'{model.__name__} supports {model.component_name}')
# Extract models that is of our interest, e.g., DRAM and SRAM models
DRAM_models_keywords = ["ddr", "dram"]
SRAM_models_keywords = ["sram"]
MAC_models_keywords = ["mac", "gemm", "float", "fp"]
DRAM_models = []
SRAM_models = []
MAC_models = []
for model in all_installed_models:
    if any(keyword in model.__name__.lower() for keyword in DRAM_models_keywords):
        DRAM_models.append(model)
    if any(keyword in model.__name__.lower() for keyword in SRAM_models_keywords):
        SRAM_models.append(model)
    if any(keyword in model.__name__.lower() for keyword in MAC_models_keywords):
        MAC_models.append(model)
# DRAM models
print("-"*20+"DRAM models"+"-"*20)
for model in DRAM_models:
    print(f'{model.__name__} supports {model.component_name}')
# SRAM models
print("-"*20+"SRAM models"+"-"*20)
for model in SRAM_models:
    print(f'{model.__name__} supports {model.component_name}')
# MAC models
print("-"*20+"MAC models"+"-"*20)
for model in MAC_models:
    print(f'{model.__name__} supports {model.component_name}')

Listing component models
Number of available component models: 75
All available component models: ['ADC', 'ADC', 'Adder', 'AdderTree', 'AladdinAdder', 'AladdinComparator', 'AladdinCounter', 'AladdinIntMAC', 'AladdinMultiplier', 'AladdinRegister', 'AtomlayerADC', 'AtomlayerDAC', 'AtomlayerEDRAM', 'AtomlayerEDRAMBus', 'AtomlayerInputBufferTransfers', 'AtomlayerRegisterLadder', 'AtomlayerRouter', 'AtomlayerShiftAdd', 'BrahmsDAC', 'Cache', 'ColDrivers', 'DDR3', 'DummyCompute', 'DummyMemory', 'DummyNetwork', 'DummyStorage', 'FlipFlop', 'FormsADC', 'FormsDAC', 'HBM2', 'HBM3', 'HBM4', 'IsaacADC', 'IsaacChip2ChipLink', 'IsaacDAC', 'IsaacEDRAM', 'IsaacEDRAMBus', 'IsaacRouter', 'IsaacRouterSharedByFour', 'IsaacShiftAdd', 'JiaDatapath', 'JiaShiftAdd', 'JiaZeroGate', 'LPDDR', 'LPDDR4', 'MaxPool', 'MemoryCell', 'Mux', 'NANDGate', 'NORGate', 'NOTGate', 'NewtonADC', 'NewtonDAC', 'NewtonEDRAM', 'NewtonEDRAMBus', 'NewtonRouter', 'NewtonShiftAdd', 'RaaamEDRAM', 'RaellaQuantMultiplier', 'RowDrivers', 'SR

In [2]:
# Find an SRAM model
print(f'\nListing SRAM models')
sram_models = hwc.get_models(name_must_include='sram')
print(f'Number of SRAM models: {len(sram_models)}')
for model in sram_models[:5]:
    print(f'{model.__name__} supports {model.component_name}')

# Grab the CACI SRAM model & use the "help" function to see its documentation
models = [s for s in sram_models if "hwcomponents_cacti" in s.__module__]
assert len(models) == 1, \
    f"Excected 1 CACTI SRAM model, got {len(models)}. Is hwcomponents_cacti installed?"
sram = models[0]
help(sram)


Listing SRAM models
Number of SRAM models: 2
SRAM supports ['SRAM', 'sram']
SmartBufferSRAM supports ['smart_buffer_sram', 'smartbuffer_sram', 'smartbuffersram']
Help on class SRAM in module hwcomponents_cacti.hwcomponents_cacti:

class SRAM(_Memory)
 |  SRAM(tech_node: float, width: int | None = None, depth: int | None = None, size: int | None = None, n_rw_ports: int = 1, n_banks=1)
 |
 |  SRAM model using CACTI.
 |
 |  Parameters
 |  ----------
 |      tech_node: The technology node of the SRAM in meters.
 |      width: The width of the read and write ports in bits. This is the number of bits
 |          that are accssed by any one read/write. Total size = width * depth.
 |      depth: The number of entries in the SRAM, each with `width` bits. Total size =
 |          width * depth. Either this or depth must be provided, but not both.
 |      size: The total size of the SRAM in bits. If provided, depth will be calculated
 |          as size / width. Either this or depth must be prov

In [3]:
# Now that we have a component model, we can use it to create components.
from hwcomponents_cacti.hwcomponents_cacti import SRAM

# Create an SRAM component using the CACTI model
sram = SRAM (
    tech_node=16e-9,
    width=1024,
    depth=256,
    size=1024*256,  # in bits
    n_rw_ports=1,
    n_banks=1
)
print(f'SRAM read energy: {sram.read(bits_per_action=8)} J')
print(f'SRAM write energy: {sram.write(bits_per_action=8)} J')
print(f'SRAM area: {sram.area} m^2')
print(f'SRAM leak power: {sram.leak_power} W')

SRAM read energy: EnergyLatency(energy=1.8941884269838074e-13, latency=5.488454545454546e-12) J
SRAM write energy: EnergyLatency(energy=2.7776189078152096e-13, latency=5.488454545454546e-12) J
SRAM area: 3.198538181818182e-08 m^2
SRAM leak power: 3.4692646522688153e-06 W


In [4]:
# Method 1: Import the model from a module and use it directly.
from hwcomponents_cacti import SRAM
sram = SRAM(
    tech_node=40e-9, # 40nm
    width=64,
    depth=1024,
    size=64*1024,  # in bits
    n_rw_ports=1,
    n_banks=1
)
read_energy, read_latency = sram.read()
print(f"SRAM read energy is {read_energy:.2e}J and read latency is {read_latency:.2e}s")
print(f"SRAM area is {sram.area:.2e}m^2. Leak power is {sram.leak_power:.2e}W")

SRAM read energy is 5.10e-12J and read latency is 8.15e-10s
SRAM area is 2.13e-08m^2. Leak power is 3.31e-07W


In [7]:
# Method 2: Ask hwcomponents to select the best model for a given component.
model = hwc.get_model(
    component_name="SRAM", # These are NOT case sensitive.
    component_attributes={
        "tech_node": 40e-9, # 40nm
        "width": 64,
        "depth": 1024,
        "size": 64*1024,
    },
    required_actions=["read"]
)
read_energy, read_latency = model.read()
print(f'Read energy is {read_energy:.2e}J and read latency is {read_latency:.2e}s')
print(f'Area is {model.area:.2e}m^2. Leak power is {model.leak_power:.2e}W')

# Method 3: Ask for specific properties from hwcomponents
attributes = {
    "tech_node": 40e-9, # 40nm
    "width": 64,
    "depth": 1024,
    "size": 64*1024,
}

read_energy = hwc.get_energy(
    component_name="SRAM",
    component_attributes=attributes,
    action_name="read",
    action_arguments={}
)
read_latency = hwc.get_latency(
    component_name="SRAM",
    component_attributes=attributes,
    action_name="read",
    action_arguments={}
)
area = hwc.get_area(
    component_name="SRAM",
    component_attributes=attributes,
)
leak_power = hwc.get_leak_power(
    component_name="SRAM",
    component_attributes=attributes,
)
print(f'Read energy is {read_energy:.2e}J and read latency is {read_latency:.2e}s')
print(f'Area is {area:.2e}m^2. Leak power is {leak_power:.2e}W')

Read energy is 5.10e-12J and read latency is 8.15e-10s
Area is 2.13e-08m^2. Leak power is 3.31e-07W
Read energy is 5.10e-12J and read latency is 8.15e-10s
Area is 2.13e-08m^2. Leak power is 3.31e-07W
